# Big-Data Image Analysis (UMAP / t-SNE / TMAP)

Dieses Notebook visualisiert große Bilddatenmengen anhand **reduzierter Deep-Embeddings** (128D) aus unserer SQLite-Datenbank.

**Ziel:** Ähnliche Bilder liegen in der Projektion **nah** beieinander, unähnliche **weit** auseinander. Wir vergleichen UMAP, t‑SNE und (optional) TMAP. Das Laden der Embeddings erfolgt **stromweise** aus SQLite (\`fetchmany()\`) mit optionalem **Subsampling**.


In [ ]:
# Install in *this* Jupyter kernel
#%pip install --upgrade pip setuptools wheel

# Pflicht:
#%pip install tqdm

# UMAP (falls verfügbar; probier zuerst ohne Pins)
#%pip install umap-learn

# Optional (nur wenn du's wirklich brauchst; kann frickelig sein)
# %pip install tmap

     |████████████████████████████████| 1.8 MB 2.7 MB/s eta 0:00:01
     |████████████████████████████████| 1.2 MB 8.1 MB/s eta 0:00:01
     |████████████████████████████████| 72 kB 2.4 MB/s eta 0:00:01
  Attempting uninstall: setuptools
    Found existing installation: setuptools 58.0.4
    Uninstalling setuptools-58.0.4:
      Successfully uninstalled setuptools-58.0.4
  Attempting uninstall: pip
    Found existing installation: pip 21.2.4
    Uninstalling pip-21.2.4:
      Successfully uninstalled pip-21.2.4
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 8.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.8/28.8 MB 8.9 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 9.3 MB/s  0:00:01eta 0:00:01
  Attempting uninstall: scikit-learn━━━━━━━━━━━━ 0/5 [llvmlite]
    Found existing installati

In [7]:
# Imports & Versionsausgabe
import sqlite3
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import json, os, math, random, time

# Optionale Pakete
HAVE_TQDM = False
try:
    from tqdm import tqdm  # type: ignore
    HAVE_TQDM = True
except Exception as _e:
    tqdm = None

HAVE_UMAP = False
UMAP_IMPORT_ERROR = None
try:
    import umap  # umap-learn
    HAVE_UMAP = True
except Exception as e:
    UMAP_IMPORT_ERROR = str(e)

HAVE_SKLEARN = False
SKLEARN_IMPORT_ERROR = None
PCA = None
TSNE = None
try:
    from sklearn.decomposition import PCA as _PCA  # type: ignore
    from sklearn.manifold import TSNE as _TSNE     # type: ignore
    PCA, TSNE = _PCA, _TSNE
    HAVE_SKLEARN = True
except Exception as e:
    SKLEARN_IMPORT_ERROR = str(e)

HAVE_TMAP = False
TMAP_IMPORT_ERROR = None
try:
    import tmap as tm  # optional, API variiert je nach Version
    HAVE_TMAP = True
except Exception as e:
    TMAP_IMPORT_ERROR = str(e)

# Seeds setzen
np.random.seed(42)
random.seed(42)

print("Versionsinfo:")
print("  numpy        ", np.__version__)
print("  matplotlib   ", mpl.__version__)
print("  sqlite3      ", sqlite3.sqlite_version)
print("  tqdm         ", "OK" if HAVE_TQDM else "nicht gefunden")
print("  umap-learn   ", getattr(umap, "__version__", "OK") if HAVE_UMAP else f"nicht gefunden ({UMAP_IMPORT_ERROR})")
print("  scikit-learn ", PCA.__module__.split('.')[0] if HAVE_SKLEARN else f"nicht gefunden ({SKLEARN_IMPORT_ERROR})")
print("  tmap         ", getattr(tm, "__version__", "OK") if HAVE_TMAP else f"nicht gefunden ({TMAP_IMPORT_ERROR})")

# Matplotlib-Defaults
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 200


Versionsinfo:
  numpy         1.24.2
  matplotlib    3.7.1
  sqlite3       3.43.2
  tqdm          OK
  umap-learn    0.5.9.post2
  scikit-learn  sklearn
  tmap          nicht gefunden (No module named 'tmap')


In [9]:
# Parameterblock (leicht änderbar)
DB_PATH = "[image_recommender.db]"  # <- anpassen
SAVE_DIR = "plots"
SAMPLE_N = 50000          # max. Punkte (Reservoir-Subsample bei Bedarf)
RANDOM_SEED = 42

USE_UMAP = True
USE_TSNE = True
USE_TMAP = False          # nur ausführen, wenn Paket vorhanden

PLOT_3D = False           # 2D default, optional 3D
COLOR_BY = "brightness"   # "brightness" | "hue" | None

print("Parameter:")
print("  DB_PATH   =", DB_PATH)
print("  SAVE_DIR  =", SAVE_DIR)
print("  SAMPLE_N  =", SAMPLE_N)
print("  SEED      =", RANDOM_SEED)
print("  UMAP/TSNE =", USE_UMAP, USE_TSNE)
print("  TMAP      =", USE_TMAP)
print("  3D        =", PLOT_3D)
print("  COLOR_BY  =", COLOR_BY)


Parameter:
  DB_PATH   = [image_recommender.db]
  SAVE_DIR  = plots
  SAMPLE_N  = 50000
  SEED      = 42
  UMAP/TSNE = True True
  TMAP      = False
  3D        = False
  COLOR_BY  = brightness


### Datenquelle
Wir verwenden **reduzierte 128D‑Embeddings** aus `advanced_features (deep_embeddings, deep_dim)` und optional **Farbwerte** aus `color_features` zur Farbcodierung (z. B. `brightness` oder ein einfacher Hue‑Proxy aus dem HSV‑Histogramm). Das Laden erfolgt **stromweise** via `fetchmany()`.

In [10]:
# Helpers
from typing import List, Tuple, Optional

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def _hue_from_h_hist(h_hist_json: str) -> float:
    """Berechne einfachen Hue-Proxy als gewichteten Mittelwert der H-Bins [0..180)."""
    try:
        hsv = json.loads(h_hist_json)
        if not isinstance(hsv, list) or len(hsv) < 1:
            return float("nan")
        H = np.array(hsv[0], dtype=np.float32)  # erste Liste: H-Kanal
        if H.size == 0:
            return float("nan")
        centers = (np.arange(H.size, dtype=np.float32) + 0.5) * (180.0 / float(H.size))
        s = float(np.sum(H))
        if s <= 0:
            return float("nan")
        return float(np.dot(H, centers) / s)  # in Grad [0, 180)
    except Exception:
        return float("nan")

def _brightness_from_stats(stats_json: str) -> float:
    try:
        cs = json.loads(stats_json)
        return float(cs.get("brightness", float("nan")))
    except Exception:
        return float("nan")

def fetch_embeddings(db_path: str,
                     sample_n: Optional[int] = None,
                     seed: int = 42,
                     chunk_size: int = 5000,
                     color_by: Optional[str] = "brightness") -> Tuple[List[str], np.ndarray, np.ndarray]:
    """
    Lädt (ids, X, cvals) stromweise via fetchmany().
    - X: float32 [N, D] (erwartet D=128)
    - cvals: Farbcodierung (brightness in [0,255] oder hue in [0,180], ggf. NaN)
    Subsampling via Reservoir-Sampling auf sample_n Elemente.
    """
    if not os.path.exists(db_path):
        print(f"[WARN] DB nicht gefunden: {db_path}")
        return [], np.empty((0,0), dtype=np.float32), np.array([], dtype=np.float32)

    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.execute(
        """
        SELECT i.image_id, a.deep_embeddings, a.deep_dim, cf.color_stats, cf.hsv_histogram
        FROM images i
        JOIN advanced_features a ON a.image_id = i.image_id
        JOIN color_features cf   ON cf.image_id = i.image_id
        WHERE a.deep_embeddings IS NOT NULL AND a.deep_dim IS NOT NULL
        """
    )

    rng = random.Random(seed)
    ids_res: List[str] = []
    emb_res: List[np.ndarray] = []
    cval_res: List[float] = []
    n_seen = 0

    use_reservoir = (sample_n is not None and sample_n > 0)
    if not use_reservoir:
        sample_n = None  # type: ignore

    pbar = tqdm(total=None, desc="Lese Embeddings", disable=(not HAVE_TQDM)) if HAVE_TQDM else None
    try:
        while True:
            rows = cur.fetchmany(chunk_size)
            if not rows:
                break
            if pbar is not None:
                pbar.update(len(rows))
            for iid, blob, d, stats_json, hsv_json in rows:
                if blob is None or d is None:
                    continue
                try:
                    d = int(d)
                except Exception:
                    continue
                arr = np.frombuffer(blob, dtype=np.float32)
                if arr.size != d:
                    continue
                # Farbwert für Farbcodierung
                if color_by == "brightness":
                    cval = _brightness_from_stats(stats_json)
                elif color_by == "hue":
                    cval = _hue_from_h_hist(hsv_json)
                else:
                    cval = float("nan")

                n_seen += 1
                if not use_reservoir:
                    # direkter Append
                    ids_res.append(iid)
                    emb_res.append(arr.copy())
                    cval_res.append(cval)
                else:
                    if len(emb_res) < int(sample_n):
                        ids_res.append(iid)
                        emb_res.append(arr.copy())
                        cval_res.append(cval)
                    else:
                        j = rng.randrange(n_seen)
                        if j < int(sample_n):
                            ids_res[j]  = iid
                            emb_res[j]  = arr.copy()
                            cval_res[j] = cval
    finally:
        if pbar is not None:
            pbar.close()
        con.close()

    if len(emb_res) == 0:
        return [], np.empty((0,0), dtype=np.float32), np.array([], dtype=np.float32)

    try:
        X = np.vstack(emb_res).astype(np.float32)
    except Exception as e:
        print("[ERROR] Konnte Embedding-Matrix nicht bauen:", e)
        return [], np.empty((0,0), dtype=np.float32), np.array([], dtype=np.float32)
    cvals = np.array(cval_res, dtype=np.float32)
    return ids_res, X, cvals

def _color_kwargs(cvals: Optional[np.ndarray], label: str):
    """Hilfsfunktion: Farbargumente für Scatter je nach Verfügbarkeit vorbereiten."""
    if cvals is None or cvals.size == 0 or not np.isfinite(cvals).any():
        return {"s": 1, "alpha": 0.5}
    vmin = np.nanmin(cvals)
    vmax = np.nanmax(cvals)
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        return {"s": 1, "alpha": 0.5}
    return {"c": cvals, "s": 1, "alpha": 0.5, "cmap": "viridis", "vmin": vmin, "vmax": vmax, "colorbar_label": label}

def scatter2d(Y: np.ndarray, cvals: Optional[np.ndarray], title: str, fname: str, color_label: str = "") -> None:
    ensure_dir(SAVE_DIR)
    plt.figure(figsize=(8, 7))
    kwargs = _color_kwargs(cvals, color_label)
    colorbar_label = kwargs.pop("colorbar_label", "") if "colorbar_label" in kwargs else ""
    sc = plt.scatter(Y[:, 0], Y[:, 1], **kwargs)
    if "c" in kwargs:
        cbar = plt.colorbar(sc, fraction=0.046, pad=0.04)
        if colorbar_label:
            cbar.set_label(colorbar_label)
    plt.title(title)
    plt.tight_layout()
    out = os.path.join(SAVE_DIR, fname)
    plt.savefig(out)
    plt.show()
    print(f"[saved] {out}")

def scatter3d(Y: np.ndarray, cvals: Optional[np.ndarray], title: str, fname: str, color_label: str = "") -> None:
    ensure_dir(SAVE_DIR)
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (import required for 3D projection)
    fig = plt.figure(figsize=(8, 7))
    ax = fig.add_subplot(111, projection='3d')
    kwargs = _color_kwargs(cvals, color_label)
    colorbar_label = kwargs.pop("colorbar_label", "") if "colorbar_label" in kwargs else ""
    sc = ax.scatter(Y[:, 0], Y[:, 1], Y[:, 2], **kwargs)
    if "c" in kwargs:
        cbar = plt.colorbar(sc, fraction=0.046, pad=0.04)
        if colorbar_label:
            cbar.set_label(colorbar_label)
    ax.set_title(title)
    plt.tight_layout()
    out = os.path.join(SAVE_DIR, fname)
    plt.savefig(out)
    plt.show()
    print(f"[saved] {out}")


### Embeddings laden

In [12]:
# Embeddings laden & berichten
from IPython.display import Markdown, display

ids, X, cvals = fetch_embeddings(DB_PATH, SAMPLE_N, RANDOM_SEED, color_by=COLOR_BY)
HAVE_DATA = (X.size > 0)

print("Geladene Punkte:", len(ids))
print("X-Shape:        ", X.shape)
if cvals is not None and cvals.size > 0:
    finite = np.isfinite(cvals)
    print("cvals:          ", cvals.shape, f"(finite: {finite.sum()} / {cvals.size})")
else:
    print("cvals:          (keine Farbcodierung)")

if not HAVE_DATA:
    display(Markdown(
        "**Hinweis:** Es wurden keine Embeddings gefunden. Bitte den *Backfill* ausführen, "
        "z.B. `python EmbedBackfill.py --db image_recommender.db --batch 192 --io-workers 12 --jpeg-reduce 8`. "
        "Danach dieses Notebook erneut starten."
    ))
else:
    ensure_dir(SAVE_DIR)
    np.savez_compressed(os.path.join(SAVE_DIR, "embeddings_128d.npz"), ids=np.array(ids), X=X, cvals=cvals)
    print(f"[saved] {os.path.join(SAVE_DIR, 'embeddings_128d.npz')}")


[WARN] DB nicht gefunden: [image_recommender.db]
Geladene Punkte: 0
X-Shape:         (0, 0)
cvals:          (keine Farbcodierung)


**Hinweis:** Es wurden keine Embeddings gefunden. Bitte den *Backfill* ausführen, z.B. `python EmbedBackfill.py --db image_recommender.db --batch 192 --io-workers 12 --jpeg-reduce 8`. Danach dieses Notebook erneut starten.

### (Optional) PCA‑Vorstufe für t‑SNE

Für t‑SNE bewährt sich eine PCA‑Reduktion auf ca. 50D zur Rauschunterdrückung und Beschleunigung.

In [ ]:
# PCA -> 50D (falls scikit-learn verfügbar, TSNE aktiviert und Daten vorhanden)
X50 = None
if HAVE_DATA and USE_TSNE and HAVE_SKLEARN:
    t0 = time.time()
    print("PCA -> 50D ...")
    pca = PCA(n_components=50, random_state=RANDOM_SEED)
    X50 = pca.fit_transform(X)
    print(f"PCA fertig: Shape {X50.shape} | elapsed {time.time()-t0:.2f}s")
    ensure_dir(SAVE_DIR)
    np.savez_compressed(os.path.join(SAVE_DIR, "pca50.npz"), X50=X50)
    print(f"[saved] {os.path.join(SAVE_DIR, 'pca50.npz')}")
else:
    if not HAVE_DATA:
        print("[skip] Keine Daten geladen.")
    elif not USE_TSNE:
        print("[skip] USE_TSNE=False")
    elif not HAVE_SKLEARN:
        print(f"[skip] scikit-learn nicht verfügbar: {SKLEARN_IMPORT_ERROR}")


### UMAP (2D/3D)

In [ ]:
# UMAP 2D/3D (cosine)
if HAVE_DATA and USE_UMAP and HAVE_UMAP:
    try:
        if not PLOT_3D:
            print("UMAP 2D (cosine) ...")
            reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=RANDOM_SEED)
            t0 = time.time()
            Y2 = reducer.fit_transform(X)
            print(f"UMAP 2D fertig: {Y2.shape} | elapsed {time.time()-t0:.2f}s")
            scatter2d(Y2, cvals if COLOR_BY else None, f"UMAP 2D (cosine) — N={X.shape[0]}", "umap2d.png", COLOR_BY or "")
        else:
            print("UMAP 3D (cosine) ...")
            reducer3 = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", n_components=3, random_state=RANDOM_SEED)
            t0 = time.time()
            Y3 = reducer3.fit_transform(X)
            print(f"UMAP 3D fertig: {Y3.shape} | elapsed {time.time()-t0:.2f}s")
            scatter3d(Y3, cvals if COLOR_BY else None, f"UMAP 3D (cosine) — N={X.shape[0]}", "umap3d.png", COLOR_BY or "")
    except Exception as e:
        print("[WARN] UMAP konnte nicht berechnet werden:", e)
elif not HAVE_DATA:
    print("[skip] Keine Daten geladen.")
elif not USE_UMAP:
    print("[skip] USE_UMAP=False")
elif not HAVE_UMAP:
    print(f"[skip] umap-learn nicht verfügbar: {UMAP_IMPORT_ERROR}")


### t‑SNE (2D)

In [ ]:
# t-SNE 2D (cosine), optional mit PCA-Input
if HAVE_DATA and USE_TSNE and HAVE_SKLEARN:
    try:
        data_tsne = X50 if X50 is not None else X
        print(f"t-SNE 2D (cosine) auf Shape {data_tsne.shape} ...")
        t0 = time.time()
        tsne = TSNE(n_components=2, metric="cosine", perplexity=30, random_state=RANDOM_SEED, init="pca")
        Yt = tsne.fit_transform(data_tsne)
        print(f"t-SNE fertig: {Yt.shape} | elapsed {time.time()-t0:.2f}s")
        scatter2d(Yt, cvals if COLOR_BY else None, f"t-SNE 2D (cosine) — N={data_tsne.shape[0]}", "tsne2d.png", COLOR_BY or "")
    except Exception as e:
        print("[WARN] t-SNE konnte nicht berechnet werden:", e)
elif not HAVE_DATA:
    print("[skip] Keine Daten geladen.")
elif not USE_TSNE:
    print("[skip] USE_TSNE=False")
elif not HAVE_SKLEARN:
    print(f"[skip] scikit-learn nicht verfügbar: {SKLEARN_IMPORT_ERROR}")


### TMAP (optional für sehr große Daten)

TMAP erzeugt aus (approx.) Nachbarschaftsbeziehungen ein vergrößerbares 2D‑Layout. Die Python‑API variiert; daher ist der folgende Abschnitt **optional** und stark abhängig von der lokalen Installation.

In [ ]:
# TMAP (nur wenn USE_TMAP=True & Paket vorhanden)
if HAVE_DATA and USE_TMAP and HAVE_TMAP:
    try:
        # Hinweis: Die TMAP-API variiert. Das folgende Beispiel verwandelt jeden Vektor in eine kleine
        # Feature-Menge (Top-K Indexe) und nutzt MinHash + LSHForest. Für echte Pipelines ggf. eigene
        # Fingerprints/LSH vorbereiten.
        K = 16
        N = X.shape[0]
        print(f"TMAP-Layout mit K={K} Top-Features pro Embedding ...")

        # Top-K Indizes pro Zeile (grobe Binarisierung der größten Komponenten)
        topk_idx = np.argpartition(X, -K, axis=1)[:, -K:]

        # MinHash + LSHForest (abhängig von tmap-Version)
        try:
            # Neuere tmap-Versionen
            enc = tm.Minhash(perm=256)
            lf = tm.LSHForest(num_perm=256)
            for i in range(N):
                mh = tm.Minhash(perm=256)
                for j in topk_idx[i]:
                    mh.add(str(int(j)))
                lf.add(str(i), mh)
            lf.index()

            cfg = tm.LayoutConfiguration()
            cfg.k = 15
            cfg.min_cluster_size = 10
            layout = tm.Layout(cfg)
            x, y, s, t, _ = layout.fit(lf)
            Ymap = np.vstack([np.array(x, dtype=np.float32), np.array(y, dtype=np.float32)]).T
            scatter2d(Ymap, cvals if COLOR_BY else None, f"TMAP 2D — N={N}", "tmap2d.png", COLOR_BY or "")
        except Exception as e2:
            print("[WARN] TMAP-Flow konnte mit dieser Version nicht ausgeführt werden:", e2)
            print("      Bitte TMAP-Dokumentation Ihrer Installation prüfen. Abschnitt wird übersprungen.")
    except Exception as e:
        print("[WARN] TMAP Abschnitt übersprungen:", e)
elif not HAVE_DATA:
    print("[skip] Keine Daten geladen.")
elif not USE_TMAP:
    print("[skip] USE_TMAP=False")
elif not HAVE_TMAP:
    print(f"[skip] tmap nicht verfügbar: {TMAP_IMPORT_ERROR}")


### Diskussion (Part 2.4)

- **Allgemein:** Wir erwarten, dass semantisch oder farblich ähnliche Bilder **Cluster** bilden. Große Abstände deuten auf deutliche Unterschiede hin (z. B. Farbe, Textur, Inhalt).
- **UMAP vs. t‑SNE:**
  - UMAP ist oft schneller und erhält sowohl lokale als auch globale Strukturen passabel.
  - t‑SNE bildet sehr dichte lokale Cluster, kann aber globale Abstände weniger zuverlässig darstellen; PCA‑Vorstufe hilft.
- **TMAP:** Nützlich bei sehr großen Datensätzen mit LSH/Graph‑Struktur; erzeugt interaktive, vergrößerbare Layouts. Setup ist jedoch API‑ und versionsabhängig.
- **Grenzen:** Reduktionen sind projektionstreu nur näherungsweise; Farbdominanzen vs. semantische Ähnlichkeit können konkurrieren. Sehr heterogene Bilder fordern die Metrik (cosine) und Preprocessing heraus.


### Exporthinweise

- Alle Grafiken und Zwischendateien werden in **`SAVE_DIR`** (Standard: `plots/`) gespeichert:
  - `embeddings_128d.npz` (IDs, X, cvals)
  - `pca50.npz` (optional)
  - `umap2d.png`, optional `umap3d.png`
  - `tsne2d.png`
  - optional `tmap2d.png`
- **Parameter anpassen:**
  - `DB_PATH` (Pfad zur SQLite‑DB)
  - `SAMPLE_N` (bei RAM‑Limit verringern)
  - `COLOR_BY` (`"brightness"`, `"hue"` oder `None`)
  - `PLOT_3D=True` für 3D‑UMAP
- **Fehlende Embeddings:** Bitte zunächst den Backfill laufen lassen (z. B. `python EmbedBackfill.py ...`).
- **Performance‑Tipps:** Größere `SAMPLE_N` schrittweise testen; bei t‑SNE ggf. zuvor PCA auf 50D; bei UMAP `n_neighbors`/`min_dist` justieren.
